In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
import gc

In [2]:
gc.collect()
df1 = pd.read_csv("./df_knni_rounded.csv", sep='\t').copy().reset_index(drop = True)
df1

,manufacturing_year,battery_capacity_kwh,odometer_km,vehicle_age_years,cycle_count,battery_health_percent,state_of_charge,depth_of_discharge,state_of_health,cell_voltage_avg,...,predicted_remaining_life_cycles,battery_failure,Vehicle_Brand_Num_Value,Vehicle_Model_Num_Value,Battery_Manufacturer_Num_Value,Terrain_Type_Num_Value,Battery_Chemistry_Num_Value,Drive_Type_Num_Value,Fleet_Or_Private_Num_Value,Vehicle_Type_Num_Value
0,2019.0,82.54,98163.0,6.94,250.0,73.61,85.230,26.34,72.93,3.60246,...,401.0,0.0,12.0,24.0,1.0,2.0,1.0,2.0,1.0,2.0
1,2018.0,64.17,113600.0,7.79,256.0,85.77,49.120,44.83,86.71,3.36580,...,1094.0,0.0,0.0,49.0,8.0,3.0,4.0,0.0,1.0,3.0
2,2018.0,90.92,242253.0,7.85,556.0,72.37,40.086,33.65,72.17,3.41520,...,645.0,0.0,17.0,48.0,3.0,2.0,4.0,2.0,0.0,6.0
3,2022.0,47.35,83700.0,6.44,353.0,85.11,54.320,37.73,82.88,3.42020,...,867.0,0.0,18.0,18.0,1.0,3.0,4.0,0.0,1.0,1.0
4,2015.0,68.04,88784.0,11.13,287.0,87.40,62.730,65.29,86.17,3.45560,...,1090.0,0.0,16.0,29.0,0.0,0.0,4.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,2016.0,45.99,182311.0,10.11,817.0,86.91,30.510,42.42,84.85,2.89010,...,2691.0,0.0,2.0,40.0,1.0,2.0,0.0,1.0,1.0,1.0
199996,2015.0,67.65,275317.0,11.06,840.0,65.08,77.610,30.76,65.08,3.57070,...,420.0,0.0,8.0,0.0,4.0,3.0,4.0,0.0,1.0,0.0
199997,2019.0,98.08,158747.0,4.89,336.0,86.08,44.590,24.64,84.90,2.98750,...,3148.0,0.0,2.0,40.0,0.0,4.0,0.0,1.0,0.0,2.0
199998,2016.0,51.49,95122.0,10.06,392.0,76.61,80.700,41.93,75.50,3.56980,...,797.0,0.0,16.0,30.0,7.0,0.0,4.0,1.0,1.0,1.0


In [3]:
class KNearestNeighbours(object):
    def __init__(self, dataset, n_neighbours : int, weight : str, algorithm : str, leaf_size : int, p : int, 
                metric : str, metric_params : dict, n_jobs : int):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.n_neighbours = n_neighbours
        self.weight = weight
        self.algorithm = algorithm
        self.leaf_size = leaf_size
        self.p = p
        self.metric = metric
        self.metric_params = metric_params
        self.n_jobs = n_jobs

    def train_split(self):

        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        from sklearn.model_selection import train_test_split
        df4 = self.dataset.copy().reset_index(drop = True)
        Y_col = 'Vehicle_Brand_Num_Value'
        X_cols = df4.loc[:, df4.columns != Y_col].columns
        # Split into 80% training and 20% testing
        X_train, X_test, y_train, y_test = train_test_split(
            df4[X_cols], df4[Y_col], test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def ML_fitting(self):
        global knn_fit_global

        knn = KNeighborsClassifier(n_neighbors=self.n_neighbours, weights=self.weight, algorithm=self.algorithm, 
                                   leaf_size=self.leaf_size, p=self.p, 
                                   metric=self.metric, metric_params=self.metric_params, n_jobs=self.n_jobs)
        knn_fit = knn.fit(X_train_global, y_train_global)
        knn_fit_global = knn_fit

    def ML_prediction(self):
        global knn_y_predict_global

        gc.collect()
        knn_y_predict = knn_fit_global.predict(X_test_global)
        knn_y_predict_global = knn_y_predict

    def evaluation(self):

        from sklearn.metrics import accuracy_score
        accuracy = accuracy_score(y_test_global, knn_y_predict_global)
        print("Accuracy:", accuracy)

        from sklearn.metrics import balanced_accuracy_score
        balanced_accuracy_score = balanced_accuracy_score(y_test_global, knn_y_predict_global)
        print("balanced_accuracy_score:", balanced_accuracy_score)

        from sklearn.metrics import top_k_accuracy_score
        top_k_accuracy_score = top_k_accuracy_score(y_test_global,knn_fit_global.predict_proba(X_test_global), k = 1)
        print("top_k_accuracy_score:", top_k_accuracy_score)

        from sklearn.metrics import average_precision_score
        average_precision_score = average_precision_score(y_test_global,knn_fit_global.predict_proba(X_test_global))
        print("average_precision_score:", average_precision_score)

        from sklearn.metrics import brier_score_loss
        brier_score_loss = brier_score_loss(y_test_global,knn_fit_global.predict_proba(X_test_global))
        print("brier_score_loss:", brier_score_loss)

        from sklearn.metrics import f1_score
        f1_score = f1_score(y_test_global, knn_y_predict_global, average='macro')
        print("f1_score_macro:", f1_score)
        from sklearn.metrics import f1_score
        f1_score_1 = f1_score(y_test_global, knn_y_predict_global, average='micro')
        print("f1_score_micro:", f1_score_1)
        f1_score_2 = f1_score(y_test_global, knn_y_predict_global, average='weighted')
        print("f1_score_weighted:", f1_score_2)
        f1_score_3 = f1_score(y_test_global, knn_y_predict_global, average=None)
        print("f1_score_None:", f1_score_3)

        from sklearn.metrics import log_loss
        log_loss = log_loss(y_test_global,knn_fit_global.predict_proba(X_test_global))
        print("log_loss:", log_loss)

        from sklearn.metrics import precision_score
        precision_score_macro = precision_score(y_test_global, knn_y_predict_global, average='macro')
        print("precision_score_macro:", precision_score_macro)
        precision_score_micro = precision_score(y_test_global, knn_y_predict_global, average='micro')
        print("precision_score_micro:", precision_score_micro)
        precision_score_weighted = precision_score(y_test_global, knn_y_predict_global, average='weighted')
        print("precision_score_weighted:", precision_score_weighted)
        precision_score_None = precision_score(y_test_global, knn_y_predict_global, average=None)
        print("precision_score_None:", precision_score_None)
        precision_score_None_ZD_1 = precision_score(y_test_global, knn_y_predict_global, average=None, zero_division=1)
        print("precision_score_None_ZD_1:", precision_score_None_ZD_1)
        precision_score_None_ZD_NAN = precision_score(y_test_global, knn_y_predict_global, average=None, zero_division=np.nan)
        print("precision_score_None_ZD_NAN:", precision_score_None_ZD_NAN)

        from sklearn.metrics import recall_score
        recall_score_macro = recall_score(y_test_global, knn_y_predict_global, average='macro')
        print("recall_score_macro:", recall_score_macro)
        recall_score_micro = recall_score(y_test_global, knn_y_predict_global, average='micro')
        print("recall_score_micro:", recall_score_micro)
        recall_score_weighted = recall_score(y_test_global, knn_y_predict_global, average='weighted')
        print("recall_score_weighted:", recall_score_weighted)
        recall_score_None = recall_score(y_test_global, knn_y_predict_global, average=None)
        print("recall_score_None:", recall_score_None)
        recall_score_None_ZD_1 = recall_score(y_test_global, knn_y_predict_global, average=None, zero_division=1)
        print("recall_score_None_ZD_1:", recall_score_None_ZD_1)
        recall_score_None_ZD_NAN = recall_score(y_test_global, knn_y_predict_global, average=None, zero_division=np.nan)
        print("recall_score_None_ZD_NAN:", recall_score_None_ZD_NAN)

        from sklearn.metrics import jaccard_score
        jaccard_score_macro = jaccard_score(y_test_global, knn_y_predict_global, average='macro')
        print("jaccard_score_macro:", jaccard_score_macro)
        jaccard_score_micro = jaccard_score(y_test_global, knn_y_predict_global, average='micro')
        print("jaccard_score_micro:", jaccard_score_micro)
        jaccard_score_weighted = jaccard_score(y_test_global, knn_y_predict_global, average='weighted')
        print("jaccard_score_weighted:", jaccard_score_weighted)
        jaccard_score_None = jaccard_score(y_test_global, knn_y_predict_global, average=None)
        print("jaccard_score_None:", jaccard_score_None)

        from sklearn.metrics import roc_auc_score
        roc_auc_score_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), multi_class = 'ovr')
        print('roc_auc_score_ovr:', roc_auc_score_ovr)
        from sklearn.metrics import roc_auc_score
        roc_auc_score_ovo = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), multi_class = 'ovo')
        print('roc_auc_score_ovo:', roc_auc_score_ovo)

        from sklearn.metrics import roc_auc_score
        roc_auc_score_macro_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), average='macro',
                                           multi_class = 'ovr')
        print("roc_auc_score_macro_ovr:", roc_auc_score_macro_ovr)
        from sklearn.metrics import roc_auc_score
        roc_auc_score_macro_ovo = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), average='macro',
                                           multi_class = 'ovo')
        print("roc_auc_score_macro_ovo:", roc_auc_score_macro_ovo)
        
        roc_auc_score_weighted_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), 
                                               average='weighted', multi_class = 'ovr')
        print("roc_auc_score_weighted_ovr:", roc_auc_score_weighted_ovr)
        roc_auc_score_weighted_ovo = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), 
                                               average='weighted', multi_class = 'ovo')
        print("roc_auc_score_weighted_ovo:", roc_auc_score_weighted_ovo)
        
        roc_auc_score_None_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), average=None,
                                              multi_class = 'ovr')
        print("roc_auc_score_None_ovr:", roc_auc_score_None_ovr)

        from sklearn.metrics import d2_log_loss_score
        d2_log_loss_score = d2_log_loss_score(y_test_global, knn_fit_global.predict_proba(X_test_global),
                                              sample_weight=None, labels=None)
        print("d2_log_loss_score:", d2_log_loss_score)

        from sklearn.metrics import d2_brier_score
        d2_brier_score = d2_brier_score(y_test_global, knn_fit_global.predict_proba(X_test_global), 
                                        sample_weight=None, labels=None)
        print("d2_brier_score:", d2_brier_score)
        

    def main(self):
        gc.collect()
        self.train_split()
        self.ML_fitting()
        self.ML_prediction()
        self.evaluation()
        gc.collect()
        

In [4]:
knn = KNearestNeighbours(df1, 5, 'uniform', 'auto', 30, 2, 'minkowski', None, None)
knn.main()

Accuracy: 0.1468
balanced_accuracy_score: 0.10216671748435753
top_k_accuracy_score: 0.13635
average_precision_score: 0.07422770615511423
brier_score_loss: 1.0604000000000002
f1_score_macro: 0.09306882923722506
f1_score_micro: 0.1468
f1_score_weighted: 0.12846526005054357
f1_score_None: [0.08585432 0.09541853 0.3880048  0.07383567 0.07450093 0.04460303
 0.07068998 0.05077929 0.02087994 0.03331419 0.01960784 0.0514449
 0.26958667 0.08489723 0.02508251 0.04759544 0.30388433 0.03284072
 0.0683112  0.02024507]
log_loss: 23.023610251603845
precision_score_macro: 0.10313309831231343
precision_score_micro: 0.1468
precision_score_weighted: 0.1320938499064245
precision_score_None: [0.05710432 0.06912902 0.28949881 0.06353861 0.07045543 0.04574565
 0.06999255 0.06084337 0.03381643 0.05882353 0.03883495 0.06982759
 0.34087363 0.13649425 0.0475     0.09142857 0.32350639 0.06289308
 0.09131905 0.04103672]
precision_score_None_ZD_1: [0.05710432 0.06912902 0.28949881 0.06353861 0.07045543 0.04574565
 

In [5]:
knn_fit_global

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[float64](20,)","[ 0., 1., 2.,...,17.,18.,19.]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
